In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load the feature‑engineered dataset
df = pd.read_csv(r'..\Data\Feature_Engineering\Feature_Engineering.csv')

X = df.drop(columns=['label'])
y = df['label']

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

X shape: (10005, 8)
y distribution:
 label
0    8000
1    2005
Name: count, dtype: int64


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\ny_train distribution:")
print(y_train.value_counts())
print("\ny_test distribution:")
print(y_test.value_counts())

X_train: (8004, 8) | X_test: (2001, 8)

y_train distribution:
label
0    6400
1    1604
Name: count, dtype: int64

y_test distribution:
label
0    1600
1     401
Name: count, dtype: int64


In [34]:
from sklearn.preprocessing import StandardScaler
import joblib
import os

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_test_sc  = scaler.transform(X_test)        # transform only – no data leakage

# Save the scaler for future use (model training / inference)
os.makedirs(r'..\Models', exist_ok=True)
joblib.dump(scaler, r'..\Models\Scaler\scaler.pkl')
print("Scaler saved to ../Models/Scaler/scaler.pkl")

Scaler saved to ../Models/Scaler/scaler.pkl


In [35]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print("Before SMOTE:", dict(y_train.value_counts()))
print("After SMOTE: ", dict(pd.Series(y_train_res).value_counts()))

Before SMOTE: {0: np.int64(6400), 1: np.int64(1604)}
After SMOTE:  {1: np.int64(6400), 0: np.int64(6400)}


In [36]:
# Create directory if it doesn't exist
os.makedirs(r'..\Data\Splits', exist_ok=True)

# Training data (SMOTE'd and scaled)
pd.DataFrame(X_train_res, columns=X.columns).to_csv(
    r'..\Data\Splits\X_train.csv', index=False)
pd.Series(y_train_res, name='label').to_csv(
    r'..\Data\Splits\y_train.csv', index=False)

# Test data (original distribution, only scaled – no SMOTE)
pd.DataFrame(X_test_sc, columns=X.columns).to_csv(
    r'..\Data\Splits\X_test.csv', index=False)
pd.Series(y_test.values, name='label').to_csv(
    r'..\Data\Splits\y_test.csv', index=False)

print("✅ All splits saved to ../Data/Splits/")
print("  - X_train.csv :", X_train_res.shape[0], "rows (balanced)")
print("  - y_train.csv :", len(y_train_res), "rows")
print("  - X_test.csv  :", X_test_sc.shape[0], "rows (real test distribution)")
print("  - y_test.csv  :", len(y_test), "rows")

✅ All splits saved to ../Data/Splits/
  - X_train.csv : 12800 rows (balanced)
  - y_train.csv : 12800 rows
  - X_test.csv  : 2001 rows (real test distribution)
  - y_test.csv  : 2001 rows


In [37]:
# Reload and check (to confirm everything works)
X_train_check = pd.read_csv(r'..\Data\Splits\X_train.csv')
y_train_check = pd.read_csv(r'..\Data\Splits\y_train.csv').squeeze()
X_test_check  = pd.read_csv(r'..\Data\Splits\X_test.csv')
y_test_check  = pd.read_csv(r'..\Data\Splits\y_test.csv').squeeze()

print("Reloaded X_train shape:", X_train_check.shape)
print("Reloaded y_train class distribution:\n", y_train_check.value_counts())
print("Reloaded X_test shape:", X_test_check.shape)
print("Reloaded y_test class distribution:\n", y_test_check.value_counts())

Reloaded X_train shape: (12800, 8)
Reloaded y_train class distribution:
 label
1    6400
0    6400
Name: count, dtype: int64
Reloaded X_test shape: (2001, 8)
Reloaded y_test class distribution:
 label
0    1600
1     401
Name: count, dtype: int64
